# Conda Environment: 09_Tree
---

# Library Imports

In [1]:
# For working with phylogenetic objects
import dendropy
from dendropy import Tree, TreeList, DnaCharacterMatrix
from dendropy.calculate import treecompare

# For visualising trees
import ete3
from ete3 import Tree as eteTree
from ete3 import TreeStyle, NodeStyle,faces, AttrFace
from IPython.display import Image
from matplotlib.colors import Normalize, to_hex
from matplotlib.cm import ScalarMappable
import io

# For data management and visualisation
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# For running linear models
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import TheilSenRegressor
import scipy.stats as stats
import statsmodels.api as sm

# For shell functionality
import os

# For running R through Python
import rpy2.robjects as ro
from rpy2.robjects.packages import importr

# R libraries
importr('ape')
importr('geiger')

rpy2.robjects.packages.Package as a <module 'geiger'>

---

# Make Output Directories

In [2]:
!mkdir -p ../Figures/Tree

---

# Load Data

In [3]:
# Trait data
trait_data = pd.read_csv("../Datasets/ASV_trait_data.csv")

# Taxonomy data
taxonomy = pd.read_csv("../Preprocessing/asvcleaner/cleaned/cleaned_species_table.tsv", sep= "\t")
taxonomy = taxonomy[['ASV_ID','Phylum']]
trait_data = pd.merge(left= trait_data, right= taxonomy, how= 'left', on= 'ASV_ID')

---

# Render Unannotated Tree

In [4]:
## Create ultrametric tree
# Load tree
Tree_16S = Tree.get(path= "../Datasets/Tree/Tree.nwk", schema= 'newick')
tns = Tree_16S.taxon_namespace

# Reroot tree at midpoint
Tree_16S.reroot_at_midpoint()

# Convert to ultrametric tree
umt = eteTree(Tree_16S.as_string(schema= 'Newick')[5:])
umt.convert_to_ultrametric()

# Export ultrametric tree to file
with open('../Datasets/Tree/Ultrametric_tree.nwk', 'w') as fl:
    fl.write(umt.write())

## render tree
os.environ['QT_QPA_PLATFORM']='offscreen' # Fixes rendering killing the kernel (bug)
# set tree style 
ts = TreeStyle()
ts.mode = "c"
ts.show_branch_length = True
# render
umt.render('../Figures/Tree/Ultrametric_tree.pdf', tree_style = ts);

QStandardPaths: XDG_RUNTIME_DIR not set, defaulting to '/tmp/runtime-s05am3'


---

# Render Tree with Taxonomy Annotation

In [5]:
%matplotlib agg

# Create dictionary of ASV_IDs and taxonomy at Family level
lookup = dict(zip(trait_data['ASV_ID'],trait_data['Phylum']))

# Create colour map
colours = {Phylum: index for index,Phylum in enumerate(trait_data['Phylum'].dropna().unique())}
cmap = sns.color_palette('tab20',20)

# Add optimum data to each node as colour
for index,node in enumerate(umt.traverse()):

    if index > 0:
        nstyle = NodeStyle()
        node_name = node.name
        if node_name in lookup.keys():
            taxon = taxonomy[taxonomy['ASV_ID'] == node_name]['Phylum'].iloc[0]
            if pd.notna(taxon):
                colour = to_hex(tuple(np.round(colour,2) for colour in cmap[colours[taxon]-1]))
            else:
                colour= 'black'
            node.add_face(faces.RectFace(width= 200, height= 50, fgcolor= colour, bgcolor= colour),column= 1,position='aligned')
        node.set_style(nstyle)

## render tree
os.environ['QT_QPA_PLATFORM']='offscreen' # Fixes rendering killing the kernel (bug)

# make scale bar
#fig, ax = plt.subplots(figsize=(2.5, 1), dpi= 300)
#plt.colorbar(sm,cax= ax, orientation= 'horizontal', ticks= range(5,46,5))
#plt.title('Optimum Temperature (°C)')
#plt.tight_layout()
#plt.savefig("../Figures/Tree/Optimum_colour_bar.png")

# set tree style 
ts = TreeStyle()
ts.mode = "c"
ts.show_branch_length = True

# add colour bar
#img_face = faces.ImgFace(img_file= "../Figures/Tree/Optimum_colour_bar.png", width=2000, height=800)
#ts.legend.add_face(img_face, column=0)


# render
umt.render('../Figures/Tree/Taxonomy_tree.pdf', tree_style = ts)

# close figure
plt.close();

---

# Render Tree with Temperature Optimum Annotation

In [6]:
%matplotlib agg

lookup = dict(zip(trait_data['ASV_ID'],trait_data['Optimum']))

# Create colour map
norm = Normalize(vmin=5, vmax=45)
cmap = plt.get_cmap('coolwarm')
sm = ScalarMappable(cmap=cmap, norm=norm)

# Add optimum data to each node as colour
for index,name in enumerate(umt.traverse()):

    if index > 0:
        nstyle = NodeStyle()
        node_name = name.name
        if node_name in lookup.keys():
            colour = to_hex(tuple(np.round(colour,2) for colour in sm.to_rgba(lookup[node_name])[:-1]))
            nstyle["hz_line_color"] = colour
            nstyle["vt_line_color"] = colour
        else:
            colour = 'black'
            nstyle["vt_line_color"] = colour
            nstyle["hz_line_color"] = colour
        nstyle["hz_line_width"] = 20
        nstyle["vt_line_width"] = 20
        nstyle["size"] = 0
        name.set_style(nstyle)
    else:
        nstyle = NodeStyle()
        node_name = name.name
        nstyle["vt_line_color"] = 'black'
        nstyle["hz_line_color"] = 'black'
        nstyle["hz_line_width"] = 20
        nstyle["vt_line_width"] = 20
        nstyle["size"] = 0
        name.set_style(nstyle)

## render tree
os.environ['QT_QPA_PLATFORM']='offscreen' # Fixes rendering killing the kernel (bug)

# make scale bar
fig, ax = plt.subplots(figsize=(2.5, 1), dpi= 300)
plt.colorbar(sm,cax= ax, orientation= 'horizontal', ticks= range(5,46,5))
plt.title('Optimum Temperature (°C)')
plt.tight_layout()
plt.savefig("../Figures/Tree/Optimum_colour_bar.png")

# set tree style 
ts = TreeStyle()
ts.mode = "c"
ts.show_branch_length = True

# add colour bar
img_face = faces.ImgFace(img_file= "../Figures/Tree/Optimum_colour_bar.png", width=2000, height=800)
ts.legend.add_face(img_face, column=0)


# render
umt.render('../Figures/Tree/Optimum_temp_tree.pdf', tree_style = ts)

# close figure
plt.close();

---

# Render Tree with Niche Breadth Annotation

In [7]:
%matplotlib agg

lookup = dict(zip(trait_data['ASV_ID'],trait_data['Bn']))
plot = "Niche_breadth"

# Create colour map
scale_dict = dict(trait_data['Bn'].describe())
norm = Normalize(vmin= scale_dict['min'], vmax= scale_dict['max'])
cmap = plt.get_cmap('viridis')
sm = ScalarMappable(cmap=cmap, norm=norm)

# Add optimum data to each node as colour
for index,node in enumerate(umt.traverse()):

    if index > 0:
        nstyle = NodeStyle()
        node_name = node.name
        if node_name in lookup.keys():
            colour = to_hex(tuple(np.round(colour,2) for colour in sm.to_rgba(lookup[node_name])[:-1]))
            nstyle["hz_line_color"] = colour
            nstyle["vt_line_color"] = colour
        else:
            colour = 'black'
            nstyle["vt_line_color"] = colour
            nstyle["hz_line_color"] = colour
        nstyle["hz_line_width"] = 20
        nstyle["vt_line_width"] = 20
        nstyle["size"] = 0
        node.set_style(nstyle)
    else:
        nstyle = NodeStyle()
        node_name = name.name
        nstyle["vt_line_color"] = 'black'
        nstyle["hz_line_color"] = 'black'
        nstyle["hz_line_width"] = 20
        nstyle["vt_line_width"] = 20
        nstyle["size"] = 0
        name.set_style(nstyle)

## render tree
os.environ['QT_QPA_PLATFORM']='offscreen' # Fixes rendering killing the kernel (bug)

# make scale bar
fig, ax = plt.subplots(figsize=(2.5, 1), dpi= 300)
plt.colorbar(sm,cax= ax, orientation= 'horizontal', ticks= [0.112,np.round(scale_dict['50%'],3),np.round(scale_dict['max'],3)])
plt.title("Niche breadth")
plt.tight_layout()
plt.savefig(f"../Figures/Tree/{plot}_colour_bar.png")

# set tree style 
ts = TreeStyle()
ts.mode = "c"
ts.show_branch_length = True

# add colour bar
img_face = faces.ImgFace(img_file= f"../Figures/Tree/{plot}_colour_bar.png", width=2000, height=800)
ts.legend.add_face(img_face, column=0)


# render
umt.render(f"../Figures/Tree/{plot}_tree.pdf", tree_style = ts)

# close figure
plt.close();

---

# Render Tree with Fitness Annotation

## Stable High Temperature Fitness

In [8]:
%matplotlib agg

lookup = dict(zip(trait_data['ASV_ID'],trait_data['highT_fitness']))
plot = "highT"

# Create colour map
#scale_dict = dict(trait_data['highT_fitness'].describe()) # Not used here
norm = Normalize(vmin= trait_data['highT_fitness'].min(), vmax=trait_data['highT_fitness'].max())
cmap = plt.get_cmap('brg')
sm = ScalarMappable(cmap=cmap, norm=norm)

# Add optimum data to each node as colour
for index,node in enumerate(umt.traverse()):

    if index > 0:
        nstyle = NodeStyle()
        node_name = node.name
        if node_name in lookup.keys():
            colour = to_hex(tuple(np.round(colour,2) for colour in sm.to_rgba(lookup[node_name])[:-1]))
            nstyle["hz_line_color"] = colour
            nstyle["vt_line_color"] = colour
        else:
            colour = 'black'
            nstyle["vt_line_color"] = colour
            nstyle["hz_line_color"] = colour
        nstyle["hz_line_width"] = 20
        nstyle["vt_line_width"] = 20
        nstyle["size"] = 0
        node.set_style(nstyle)
    else:
        nstyle = NodeStyle()
        node_name = name.name
        nstyle["vt_line_color"] = 'black'
        nstyle["hz_line_color"] = 'black'
        nstyle["hz_line_width"] = 20
        nstyle["vt_line_width"] = 20
        nstyle["size"] = 0
        name.set_style(nstyle)

## render tree
os.environ['QT_QPA_PLATFORM']='offscreen' # Fixes rendering killing the kernel (bug)

# make scale bar
fig, ax = plt.subplots(figsize=(2.5, 1), dpi= 300)
plt.colorbar(sm,cax= ax, orientation= 'horizontal')
plt.title("Fitness")
plt.tight_layout()
plt.savefig(f"../Figures/Tree/{plot}_colour_bar.png")

# set tree style 
ts = TreeStyle()
ts.mode = "c"
ts.show_branch_length = True

# add colour bar
img_face = faces.ImgFace(img_file= f"../Figures/Tree/{plot}_colour_bar.png", width=2000, height=800)
ts.legend.add_face(img_face, column=0)


# render
umt.render(f"../Figures/Tree/{plot}_tree.pdf", tree_style = ts)

# close figure
plt.close();

## Stable Low Temperature Fitness

In [9]:
%matplotlib agg

lookup = dict(zip(trait_data['ASV_ID'],trait_data['lowT_fitness']))
plot = "lowT"

# Create colour map
norm = Normalize(vmin= trait_data['lowT_fitness'].min(), vmax=trait_data['lowT_fitness'].max())
cmap = plt.get_cmap('brg')
sm = ScalarMappable(cmap=cmap, norm=norm)

# Add optimum data to each node as colour
for index,node in enumerate(umt.traverse()):

    if index > 0:
        nstyle = NodeStyle()
        node_name = node.name
        if node_name in lookup.keys():
            colour = to_hex(tuple(np.round(colour,2) for colour in sm.to_rgba(lookup[node_name])[:-1]))
            nstyle["hz_line_color"] = colour
            nstyle["vt_line_color"] = colour
        else:
            colour = 'black'
            nstyle["vt_line_color"] = colour
            nstyle["hz_line_color"] = colour
        nstyle["hz_line_width"] = 20
        nstyle["vt_line_width"] = 20
        nstyle["size"] = 0
        node.set_style(nstyle)
    else:
        nstyle = NodeStyle()
        node_name = name.name
        nstyle["vt_line_color"] = 'black'
        nstyle["hz_line_color"] = 'black'
        nstyle["hz_line_width"] = 20
        nstyle["vt_line_width"] = 20
        nstyle["size"] = 0
        name.set_style(nstyle)

## render tree
os.environ['QT_QPA_PLATFORM']='offscreen' # Fixes rendering killing the kernel (bug)

# make scale bar
fig, ax = plt.subplots(figsize=(2.5, 1), dpi= 300)
plt.colorbar(sm,cax= ax, orientation= 'horizontal')
plt.title("Fitness")
plt.tight_layout()
plt.savefig(f"../Figures/Tree/{plot}_colour_bar.png")

# set tree style 
ts = TreeStyle()
ts.mode = "c"
ts.show_branch_length = True

# add colour bar
img_face = faces.ImgFace(img_file= f"../Figures/Tree/{plot}_colour_bar.png", width=2000, height=800)
ts.legend.add_face(img_face, column=0)


# render
umt.render(f"../Figures/Tree/{plot}_tree.pdf", tree_style = ts)

# close figure
plt.close();

## Two Week Fluctuation Fitness

In [10]:
%matplotlib agg

lookup = dict(zip(trait_data['ASV_ID'],trait_data['TwoWeeks_fitness']))
plot = "TwoWeeks"

# Create colour map
norm = Normalize(vmin= trait_data['TwoWeeks_fitness'].min(), vmax=trait_data['TwoWeeks_fitness'].max())
cmap = plt.get_cmap('brg')
sm = ScalarMappable(cmap=cmap, norm=norm)

# Add optimum data to each node as colour
for index,node in enumerate(umt.traverse()):

    if index > 0:
        nstyle = NodeStyle()
        node_name = node.name
        if node_name in lookup.keys():
            colour = to_hex(tuple(np.round(colour,2) for colour in sm.to_rgba(lookup[node_name])[:-1]))
            nstyle["hz_line_color"] = colour
            nstyle["vt_line_color"] = colour
        else:
            colour = 'black'
            nstyle["vt_line_color"] = colour
            nstyle["hz_line_color"] = colour
        nstyle["hz_line_width"] = 20
        nstyle["vt_line_width"] = 20
        nstyle["size"] = 0
        node.set_style(nstyle)
    else:
        nstyle = NodeStyle()
        node_name = name.name
        nstyle["vt_line_color"] = 'black'
        nstyle["hz_line_color"] = 'black'
        nstyle["hz_line_width"] = 20
        nstyle["vt_line_width"] = 20
        nstyle["size"] = 0
        name.set_style(nstyle)

## render tree
os.environ['QT_QPA_PLATFORM']='offscreen' # Fixes rendering killing the kernel (bug)

# make scale bar
fig, ax = plt.subplots(figsize=(2.5, 1), dpi= 300)
plt.colorbar(sm,cax= ax, orientation= 'horizontal')
plt.title("Fitness")
plt.tight_layout()
plt.savefig(f"../Figures/Tree/{plot}_colour_bar.png")

# set tree style 
ts = TreeStyle()
ts.mode = "c"
ts.show_branch_length = True

# add colour bar
img_face = faces.ImgFace(img_file= f"../Figures/Tree/{plot}_colour_bar.png", width=2000, height=800)
ts.legend.add_face(img_face, column=0)


# render
umt.render(f"../Figures/Tree/{plot}_tree.pdf", tree_style = ts)

# close figure
plt.close();

## One Week Fluctuation Fitness

In [11]:
%matplotlib agg

lookup = dict(zip(trait_data['ASV_ID'],trait_data['OneWeek_fitness']))
plot = "OneWeek"

# Create colour map
norm = Normalize(vmin= trait_data['OneWeek_fitness'].min(), vmax=trait_data['OneWeek_fitness'].max())
cmap = plt.get_cmap('brg')
sm = ScalarMappable(cmap=cmap, norm=norm)

# Add optimum data to each node as colour
for index,node in enumerate(umt.traverse()):

    if index > 0:
        nstyle = NodeStyle()
        node_name = node.name
        if node_name in lookup.keys():
            colour = to_hex(tuple(np.round(colour,2) for colour in sm.to_rgba(lookup[node_name])[:-1]))
            nstyle["hz_line_color"] = colour
            nstyle["vt_line_color"] = colour
        else:
            colour = 'black'
            nstyle["vt_line_color"] = colour
            nstyle["hz_line_color"] = colour
        nstyle["hz_line_width"] = 20
        nstyle["vt_line_width"] = 20
        nstyle["size"] = 0
        node.set_style(nstyle)
    else:
        nstyle = NodeStyle()
        node_name = name.name
        nstyle["vt_line_color"] = 'black'
        nstyle["hz_line_color"] = 'black'
        nstyle["hz_line_width"] = 20
        nstyle["vt_line_width"] = 20
        nstyle["size"] = 0
        name.set_style(nstyle)

## render tree
os.environ['QT_QPA_PLATFORM']='offscreen' # Fixes rendering killing the kernel (bug)

# make scale bar
fig, ax = plt.subplots(figsize=(2.5, 1), dpi= 300)
plt.colorbar(sm,cax= ax, orientation= 'horizontal')
plt.title("Fitness")
plt.tight_layout()
plt.savefig(f"../Figures/Tree/{plot}_colour_bar.png")

# set tree style 
ts = TreeStyle()
ts.mode = "c"
ts.show_branch_length = True

# add colour bar
img_face = faces.ImgFace(img_file= f"../Figures/Tree/{plot}_colour_bar.png", width=2000, height=800)
ts.legend.add_face(img_face, column=0)


# render
umt.render(f"../Figures/Tree/{plot}_tree.pdf", tree_style = ts)

# close fiture
plt.close();

## Two Day Fluctuation Fitness

In [12]:
%matplotlib agg

lookup = dict(zip(trait_data['ASV_ID'],trait_data['TwoDays_fitness']))
plot = "TwoDays"

# Create colour map
norm = Normalize(vmin= trait_data['TwoDays_fitness'].min(), vmax=trait_data['TwoDays_fitness'].max())
cmap = plt.get_cmap('brg')
sm = ScalarMappable(cmap=cmap, norm=norm)

# Add optimum data to each node as colour
for index,node in enumerate(umt.traverse()):

    if index > 0:
        nstyle = NodeStyle()
        node_name = node.name
        if node_name in lookup.keys():
            colour = to_hex(tuple(np.round(colour,2) for colour in sm.to_rgba(lookup[node_name])[:-1]))
            nstyle["hz_line_color"] = colour
            nstyle["vt_line_color"] = colour
        else:
            colour = 'black'
            nstyle["vt_line_color"] = colour
            nstyle["hz_line_color"] = colour
        nstyle["hz_line_width"] = 20
        nstyle["vt_line_width"] = 20
        nstyle["size"] = 0
        node.set_style(nstyle)
    else:
        nstyle = NodeStyle()
        node_name = name.name
        nstyle["vt_line_color"] = 'black'
        nstyle["hz_line_color"] = 'black'
        nstyle["hz_line_width"] = 20
        nstyle["vt_line_width"] = 20
        nstyle["size"] = 0
        name.set_style(nstyle)

## render tree
os.environ['QT_QPA_PLATFORM']='offscreen' # Fixes rendering killing the kernel (bug)

# make scale bar
fig, ax = plt.subplots(figsize=(2.5, 1), dpi= 300)
plt.colorbar(sm,cax= ax, orientation= 'horizontal')
plt.title("Fitness")
plt.tight_layout()
plt.savefig(f"../Figures/Tree/{plot}_colour_bar.png")

# set tree style 
ts = TreeStyle()
ts.mode = "c"
ts.show_branch_length = True

# add colour bar
img_face = faces.ImgFace(img_file= f"../Figures/Tree/{plot}_colour_bar.png", width=2000, height=800)
ts.legend.add_face(img_face, column=0)


# render
umt.render(f"../Figures/Tree/{plot}_tree.pdf", tree_style = ts)

# close figure
plt.close();